# ScopeOne Minimal Example

This notebook shows the minimum external workflow:

- import `ScopeOne`
- load a Micro-Manager config
- query available cameras
- start preview
- grab one frame into the notebook and display it inline
- stop preview and unload the config

Start `ScopeOne.exe` first. This notebook connects to the local ScopeOne API server.


In [ ]:
import sys
from pathlib import Path

import numpy as np
from IPython.display import display
from PIL import Image


def add_scopeone_python_path() -> tuple[Path, Path]:
    here = Path.cwd().resolve()
    candidates = [here, *here.parents]
    for base in candidates:
        if (base / "src" / "scopeone" / "__init__.py").exists() and (base / "pyproject.toml").exists():
            src_dir = base / "src"
            if str(src_dir) not in sys.path:
                sys.path.insert(0, str(src_dir))
            return base, base.parents[2]
    raise RuntimeError("Cannot find ScopeOne python project root")


def to_display_image(frame: np.ndarray) -> np.ndarray:
    frame = np.asarray(frame)
    if frame.dtype == np.uint8:
        return frame

    frame_min = int(frame.min())
    frame_max = int(frame.max())
    if frame_max <= frame_min:
        return np.zeros(frame.shape, dtype=np.uint8)

    scaled = (frame.astype(np.float32) - frame_min) / (frame_max - frame_min)
    return np.clip(scaled * 255.0, 0, 255).astype(np.uint8)


project_root, repo_root = add_scopeone_python_path()
config_path = repo_root / "config" / "MMConfig_demo.cfg"

from scopeone import ScopeOne

print("project_root:", project_root)
print("config_path:", config_path)
print("Make sure ScopeOne.exe is already running.")


In [ ]:
scopeone = ScopeOne()
scopeone.load_config(str(config_path))

camera_ids = scopeone.camera_ids()
print("cameras:", camera_ids)

if not camera_ids:
    raise RuntimeError("No cameras available after loading config")

camera_id = camera_ids[0]
print("using camera:", camera_id)


In [ ]:
scopeone.start_preview(camera_id)
print(f"Preview started for {camera_id}.")


In [ ]:
session = scopeone.record(frames=1, camera=camera_id)
frame = session.frame(camera_id, 0)

print("frame shape:", frame.shape)
print("frame dtype:", frame.dtype)

display_frame = to_display_image(frame)
display(Image.fromarray(display_frame, mode="L"))


In [ ]:
scopeone.stop_preview(camera_id)
scopeone.unload_config()
print(f"Preview stopped for {camera_id}.")
print("Configuration unloaded.")
